In [1]:
import torch 
import numpy as np 
import h5py
import os
from pathlib import Path
import importlib
import IPython.display as ipd
from corpus.swc_mono_test import SWCMonoTestSetH5Dataset

from corpus.speaker_room_dataset import SpeakerRoomDataset
import src.audio_transforms as at
import scipy.stats as stats

import src.spatial_attn_lightning as binaural_lightning 
import yaml
from pytorch_lightning import Trainer
from tqdm.auto import tqdm
import src.spatial_attn_architecture as attn_arch
import re
import src.custom_modules as cm

os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"

# torch.set_float32_matmul_precision('medium')
# torch.backends.cuda.matmul.allow_tf32 = True
# torch.backends.cudnn.allow_tf32 = True

/om2/user/imgriff/conda_envs/pytorch_2/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
!hostname

node103


In [3]:
### Get most recent config
# config_path = "config/binaural_attn/word_task_25p_loc_v07_LN_last_valid_time_no_affine.yaml"
# ckpt_path = "attn_cue_models/word_task_25p_loc_v07_LN_last_valid_time_no_affine/checkpoints/epoch=3-step=49432.ckpt"
# old_style = False
### Get most recent config
model_name = "word_task_v10_main_feature_gain_config"
config_path = f"config/binaural_attn/{model_name}.yaml"
ckpt_path = f"attn_cue_models/{model_name}/checkpoints/epoch=1-step=24679.ckpt"

old_style = False 

config = yaml.load(open(config_path, 'r'), Loader=yaml.FullLoader)

config['hparas']['batch_size'] = 2 # config['data']['loader']['batch_size'] // args.gpus
config['num_workers'] = 2
config['noise_kwargs']['low_snr'] = 0
config['noise_kwargs']['high_snr'] = 0
# get model input sr for brir resampling
signal_sr = config['audio']['rep_kwargs']['sr']
coch_sr = config['audio']['rep_kwargs']['env_sr']
cue_duration = .25
n_cue_frames = int(cue_duration * signal_sr)
config['model']['n_cue_frames'] = n_cue_frames

config['cue_duration_test'] = True


In [4]:
model_name

# get version str from model name 
import re 

int(re.search("v\d+", model_name).group(0).strip('v'))

10

In [5]:
def reformat_state_dict(checkpoint, ln_affine=True):
    new_state_dict = {}
    for k,v in  checkpoint['state_dict'].items():
        if ln_affine:
            # update key for easy norm layer access
            if 'conv' in k and '.0.' in k:
                k = k.replace('conv_block_', 'layer_norm_')
                k = k.replace('.0.', '.')
            # decrement conv layer ixs in dict to match model
            elif 'conv' in k and '.1.' in k:
                k = k.replace('.1.', '.0.')
            # decrement pool layer rixs in dict to match model
            elif 'conv' in k and '.3.' in k:
                k = k.replace('.3.', '.2.')
        if 'coch' in k:
            continue
        new_state_dict[k] = v

    return new_state_dict

In [6]:
import torch.nn as nn
from collections import OrderedDict
from src.layers import conv2d_same
from src.layers import padding as pad_utils
from src.custom_modules import HannPooling2d
from src.spatial_attn_architecture import BinauralAuditoryAttentionCNN

class SimpleAttentionalGain(nn.Module):
    def __init__(self, frequency_dim, cnn_channels, global_avg_cue=False, n_cue_frames=None, additive=False, **kwargs):
        super(SimpleAttentionalGain, self).__init__()
        self.frequency_dim = frequency_dim
        if global_avg_cue:
            self.ouptut_shape = (1,1)
            # self.time_average = nn.AdaptiveAvgPool2d((1, 1)) # outsize is N, C, 1, 1
        else:
            self.ouptut_shape = (frequency_dim, 1)
            # self.time_average = nn.AdaptiveAvgPool2d((frequency_dim, 1)) # outsize is N, C, FreqDim, 1
        self.bias = nn.Parameter(torch.zeros(1)) # init gain scaling to zero
        self.slope = nn.Parameter(torch.ones(1)) # init slope to one
        self.threshold = nn.Parameter(torch.zeros(1)) # init threshold to zero
        self.n_cue_frames = n_cue_frames # duration of cue in frames - full cue if None
        self.additive = additive # if True, gain is added to mixture, else multiplied
        if self.n_cue_frames:
            if n_cue_frames < 0:
                self.n_cue_frames = 1 
            print(f"Using cue duration of {self.n_cue_frames} frames")
        self.reset_parameters() 

    def reset_parameters(self):
        nn.init.constant_(self.bias, 0)
        nn.init.constant_(self.slope, 1)
        nn.init.constant_(self.threshold, 0)

    def forward(self, cue, mixture, cue_mask_ixs):
        ## Process cue 
        # time average - same as nn op, but is compat with compile 
        # get middle frames if n_cue_frames is set
        if self.n_cue_frames:
            n_total_frames =  cue.size(-1)
            start = int((n_total_frames-self.n_cue_frames)/2)
            end = start + self.n_cue_frames
            cue = cue[...,  start : end]
        cue = cue.mean(axis=-1,keepdim=True)
        # cue = self.time_average(cue)
        # apply threshold shift
        cue = cue - self.threshold
        # apply slope
        cue = cue * self.slope
        # apply sigmoid & bias
        gain = self.bias + (1-self.bias) * torch.sigmoid(cue)
        ## account for no-cue examples - no gain scaling applied
        if cue_mask_ixs is not None:
            gain[cue_mask_ixs,:] = 1
        # Apply to mixture (element mult)
        if self.additive:
            mixture = torch.add(mixture, gain)
        else:
            mixture = torch.mul(mixture, gain)
        return mixture
    

class CueDurationCNNNew(BinauralAuditoryAttentionCNN):
    def __init__(self, input_sr, out_channels, kernel, stride, padding, pool_stride, pool_size, pool_padding, attn, dropout,
                  fc_size=512, global_avg_cue=False, num_classes={"num_words":800, "num_locs":504}, frequency_dim=40,
                  residual_attn=False, n_cue_frames=None, starting_output_len = 20000, norm_first=True, ln_affine=True, additive=False, **kwargs):
        # 1. call parent constructor
        super(CueDurationCNNNew, self).__init__(input_sr, out_channels, kernel, stride, padding,
                                             pool_stride, pool_size, pool_padding, attn, dropout,
                                             fc_size, global_avg_cue, num_classes, frequency_dim,
                                             residual_attn, n_cue_frames, starting_output_len, norm_first,
                                             ln_affine, **kwargs)
        self.model_dict = nn.ModuleDict()
        self.layer_norm_params = {}
        self.output_height = frequency_dim
        self.output_len = starting_output_len # softcode eventually
        self.n_cue_frames = n_cue_frames
        self.ln_cue_frames = n_cue_frames 
        print(n_cue_frames)
        if n_cue_frames < 5000:
            self.ln_cue_frames = 5000
        # build architecture without nn.normalization functions 

        # add init norm params 
        self.layer_norm_params[f'norm_coch_rep'] = {"weight": None, "bias": None, "cue_weight": None, "cue_bias": None, 'ln_cue_frames': self.ln_cue_frames}

        for idx in range(self.n_layers):
            nIn = self.input_channels if idx == 0 else out_channels[idx - 1]
            nOut = out_channels[idx]

            # Attentional block:
            if self.attn[idx] == 1:
                self.model_dict[f'attn{idx}'] = SimpleAttentionalGain(self.output_height, nIn, global_avg_cue=global_avg_cue, n_cue_frames=self.n_cue_frames, additive=additive)

            # pre-compute conv output sizes - will assign to self.output_height and self.output_len after defining block
            # Sizes will be used for normalization layers, but depend on order of norm and conv 
            # norm -> conv gets prior output shapes, conv -> norm gets new output shapes)
            # compute output shapes using conv formula [(Height + 2Pad - dilation * (kernel - 1) -1) /  Stride] + 1
            # ignoring dilation since it's not used in this model (dilation = 1)
            if self.padding[idx] == 'same':
                output_height = self.output_height
                output_len = self.output_len
            else:
                conv_padding, _ = pad_utils.get_padding_value(self.padding[idx], self.kernel[idx], stride=self.stride[idx])
                output_height = int(np.floor((self.output_height + (2 * conv_padding[0]) - (kernel[idx][0] - 1) - 1) / stride[idx][0]) + 1)
                output_len = int(np.floor((self.output_len + (2 * conv_padding[1]) -  (kernel[idx][1] - 1) - 1) / stride[idx][1]) + 1)
                n_cue_frames = int(np.floor((self.n_cue_frames + (2 * conv_padding[1]) -  (kernel[idx][1] - 1) - 1) / stride[idx][1]) + 1)
                ln_cue_frames = int(np.floor((self.ln_cue_frames + (2 * conv_padding[1]) -  (kernel[idx][1] - 1) - 1) / stride[idx][1]) + 1)

            block = nn.Sequential(conv2d_same.create_conv2d_pad(nIn, nOut, self.kernel[idx], stride=self.stride[idx], padding=self.padding[idx]),
                                  nn.ReLU())
            
            # init layer norm params as None - will populate if in checkpoint 
            self.layer_norm_params[f'layer_norm_{idx}'] = {"weight": None, "bias": None, "cue_weight": None, "cue_bias": None, 'ln_cue_frames': self.ln_cue_frames if self.norm_first else ln_cue_frames}
            # update post-conv init
            self.output_height, self.output_len, self.n_cue_frames, self.ln_cue_frames = output_height, output_len, n_cue_frames, ln_cue_frames
            self.model_dict[f'conv_block_{idx}'] = block

            if self.pool_stride[idx] != -1:
                self.model_dict[f'hann_pool_{idx}'] = HannPooling2d(stride=self.pool_stride[idx], pool_size=self.pool_size[idx], padding=self.pool_padding[idx])
                # Compute output shapes for pooling layers using conv formula [(Height - Filter + 2Pad)/ Stride]+1
                self.output_height = int(np.floor((self.output_height - pool_size[idx][0] + 2 * pool_padding[idx][0]) / pool_stride[idx][0]) + 1)
                self.output_len = int(np.floor((self.output_len - pool_size[idx][1] + 2 * pool_padding[idx][1]) / pool_stride[idx][1]) + 1)
                self.n_cue_frames = int(np.floor((self.n_cue_frames - pool_size[idx][1] + 2 * pool_padding[idx][1]) / pool_stride[idx][1]) + 1)
                self.ln_cue_frames = int(np.floor((self.ln_cue_frames - pool_size[idx][1] + 2 * pool_padding[idx][1]) / pool_stride[idx][1]) + 1)


        self.output_size = self.output_height * nOut * self.output_len
        self.fullyconnected = nn.Linear(self.output_size, fc_size)
        self.relufc = nn.ReLU()
        self.dropout = nn.Dropout(dropout)
        # init ln params

        if self.dual_task:
            self.classificationWord = nn.Linear(fc_size, self.num_words)
            self.classificationLoc = nn.Linear(fc_size, self.num_locs)
        else:
            self.classification = nn.Linear(fc_size, self.num_classes)
    
    def forward(self, cue=None, mixture=None, cue_mask_ixs=None):
        cue = nn.functional.layer_norm(cue, cue.shape[1:],
                                       weight=self.layer_norm_params['norm_coch_rep']['cue_weight'], # [..., :  self.layer_norm_params[f'norm_coch_rep']['n_cue_frames']]
                                       bias=self.layer_norm_params['norm_coch_rep']['cue_bias'])
        
        attn = nn.functional.layer_norm(mixture, mixture.shape[1:], 
                                        weight=self.layer_norm_params['norm_coch_rep']['weight'], 
                                        bias=self.layer_norm_params['norm_coch_rep']['bias'])

        for idx in range(self.n_layers):
            if self.attn[idx] == 1:
                if self.residual_attn:
                    attn = self.model_dict[f'attn{idx}'](cue, attn, cue_mask_ixs) + attn
                else:
                    attn = self.model_dict[f'attn{idx}'](cue, attn, cue_mask_ixs)
                    
            # print(f"conv_block_{idx} post gain max and min: {attn.max().item(), attn.min().item()}")
            if self.norm_first:
                cue = nn.functional.layer_norm(cue, cue.shape[1:], 
                                               weight=self.layer_norm_params[f'layer_norm_{idx}']['cue_weight'],
                                               bias=self.layer_norm_params[f'layer_norm_{idx}']['cue_bias'])
                cue = self.model_dict[f'conv_block_{idx}'](cue)
                attn = nn.functional.layer_norm(attn, attn.shape[1:], 
                                                weight=self.layer_norm_params[f'layer_norm_{idx}']['weight'],
                                                bias=self.layer_norm_params[f'layer_norm_{idx}']['bias'])
                attn = self.model_dict[f'conv_block_{idx}'](attn)
            else:
                cue = self.model_dict[f'conv_block_{idx}'](cue)
                cue = nn.functional.layer_norm(cue, cue.shape[1:], 
                                               weight=self.layer_norm_params[f'layer_norm_{idx}']['cue_weight'],
                                               bias=self.layer_norm_params[f'layer_norm_{idx}']['cue_bias'])
                attn = self.model_dict[f'conv_block_{idx}'](attn)
                attn = nn.functional.layer_norm(attn, attn.shape[1:], 
                                                weight=self.layer_norm_params[f'layer_norm_{idx}']['weight'],
                                                bias=self.layer_norm_params[f'layer_norm_{idx}']['bias'])

            # print(f"conv_block_{idx} post conv and norm max and min: {attn.max().item(), attn.min().item()}")
            if self.pool_stride[idx] != -1:
                cue = self.model_dict[f'hann_pool_{idx}'](cue)
                attn = self.model_dict[f'hann_pool_{idx}'](attn)

        out = attn

        out = out.view(out.size(0), self.output_size) # B x FC size
        out = self.fullyconnected(out)        
        out = self.relufc(out)
        out = self.dropout(out)        
        if self.dual_task:
            word_out = self.classificationWord(out)
            loc_out = self.classificationLoc(out)
            return word_out, loc_out
        else:
            return self.classification(out)

In [7]:
# dataset = SpeakerRoomDataset(manifest_path='/om2/user/rphess/Auditory-Attention/final_binaural_manifest.pkl',
#                             excerpt_path='/om2/user/msaddler/spatial_audio_pipeline/assets/swc/manifest_all_words.pdpkl',
#                             cue_type='voice_and_location',
#                             sr=signal_sr) 

dataset = SWCMonoTestSetH5Dataset(h5_path="/om/user/imgriff/datasets/human_word_rec_SWC_2024/model_eval_stim.h5",
                                            eval_distractor_cond="1-talker-english-different",
                                            model_sr=44100,
                                            label_type='CV')
    

diotic_transforms = at.AudioCompose([
                    at.AudioToTensor(),
                    at.CombineWithRandomDBSNR(low_snr=0, high_snr=0), 
                    at.RMSNormalizeForegroundAndBackground(rms_level=0.02),  # 0.02 is the default for CV-based models 
                    at.DuplicateChannel(),

            ])


diotic_transforms = diotic_transforms.cuda()


def single_signal_collate_fn(batch):
    #apply transforsms to batch
    cues = torch.stack([diotic_transforms(cue, None)[0] for cue, fg, bg, label, confusion in batch])
    mixtures = torch.stack([diotic_transforms(fg, bg)[0] for cue, fg, bg, label, confusion in batch]).type(torch.FloatTensor)
    labels = torch.tensor([label for cue, fg, bg, label, confusion in batch]).type(torch.LongTensor)
    confusion = torch.tensor([confusion for cue, fg, bg, label, confusion in batch]).type(torch.LongTensor)
    return cues, mixtures, labels, confusion

dataloader = torch.utils.data.DataLoader(dataset, batch_size=16, shuffle=False, num_workers=config['num_workers'], collate_fn=single_signal_collate_fn)


In [28]:
class CenterCropCue(torch.nn.Module):
    """
    Center crops the cue to make a shorter signal.
    """
    def __init__(self, signal_size, crop_length, signal_sr):
        super(CenterCropCue, self).__init__()
        self.crop_length = int(crop_length * signal_sr)
        self.signal_size = int(signal_size * signal_sr)
        self.sr = signal_sr 
        self.start_crop_center = int((self.signal_size-self.crop_length)/2)

        # compute pad context for zero padding back to original length
        self.pad_context = (self.signal_size - self.crop_length) // 2

    def forward(self, cue_wav):
        """
        Args:
            cue_wav (torch.Tensor): the waveform that will be used as
                the cue audio sample (usually speech)
        """
        cue_wav = cue_wav[..., self.start_crop_center : self.start_crop_center + self.crop_length]
        # pad cue back to original length
        if cue_wav.shape[-1] < self.signal_size:
            cue_wav = torch.nn.functional.pad(cue_wav, (self.pad_context, self.pad_context), "constant", 0)
            if cue_wav.shape[-1] % 2 != 0:
                cue_wav = torch.nn.functional.pad(cue_wav, (0, 1), "constant", 0)
        return cue_wav

class CenterCropCuePad(torch.nn.Module):
    """
    Center crops the cue to make a shorter signal.
    """
    def __init__(self, signal_size, crop_length, signal_sr, pad_dur=False):
        super(CenterCropCuePad, self).__init__()
        self.crop_length = int(crop_length * signal_sr)
        self.signal_size = int(signal_size * signal_sr)
        self.sr = signal_sr 
        self.start_crop_center = int((self.signal_size-self.crop_length)/2)

        # compute pad context for zero padding back to original length
        self.pad_context = (self.signal_size - self.crop_length) // 2
        self.pad_to_dur = False 

        if pad_dur > crop_length:
            print("Padding to duration")
            self.pad_to_dur = True 
            self.pad_context = (int(pad_dur * signal_sr) - self.crop_length) // 2 

    def forward(self, cue_wav):
        """
        Args:
            cue_wav (torch.Tensor): the waveform that will be used as
                the cue audio sample (usually speech)
        """
        cue_wav = cue_wav[..., self.start_crop_center : self.start_crop_center + self.crop_length]
        # pad cue back to original length
        if self.pad_to_dur:
            cue_wav = torch.nn.functional.pad(cue_wav, (self.pad_context, self.pad_context), "constant", 0)
            if cue_wav.shape[-1] % 2 != 0:
                cue_wav = torch.nn.functional.pad(cue_wav, (0, 1), "constant", 0)
        return cue_wav
    
# torch module to center crop the cochleagram to a given duration 
class CenterCropCoch(torch.nn.Module):
    def __init__(self, crop_duration, sig_duration, sr, pad_dur=False):
        super().__init__()
        self.n_crop_frames = int(crop_duration * sr)
        sig_frames = int(sig_duration * sr)
        self.crop_context = sig_frames - self.n_crop_frames
        self.crop_start = self.crop_context // 2
        self.crop_end = self.crop_start + self.n_crop_frames
        self.pad_to_dur = False
        if pad_dur > crop_duration:
            print("Padding to duration")
            self.pad_to_dur = True 
            self.pad_context = (int(pad_dur * sr) - self.n_crop_frames) //2 

    def forward(self, x):
        # crop x to n_crop_frames and zero pad back to original length
        cropped = x[..., self.crop_start:self.crop_end]
        if self.pad_to_dur:
            cropped = torch.nn.functional.pad(cropped, (self.pad_context, self.pad_context), "constant", 0)
            if cropped.shape[-1] % 2 != 0:
                cropped = torch.nn.functional.pad(cropped, (0, 1), "constant", 0)
        return cropped 



In [29]:
!hostname

node103


In [30]:
%matplotlib inline
import matplotlib.pyplot as plt

# plt.imshow(cue[0][0].cpu().numpy(), aspect='auto', origin='upper')

In [33]:
output_dict = {'results': None, 'confusions': None}
accuracies = []
pred_list = []
true_word_int = []
confusions_list  = []
all_probs_of_interest = []
guessed_both = []

crop_duration = .25 # cue_duration # # in seconds 
sig_duration = 2.5 # in seconds 
# center_crop = CenterCropCue(crop_duration, sig_duration, signal_sr, pad_dur=2).cuda() 
# center_crop = CenterCropCue(sig_duration, crop_duration, signal_sr).cuda() 




signal_sr = config['audio']['rep_kwargs']['sr']
coch_sr = config['audio']['rep_kwargs']['env_sr']
n_cue_frames = int(crop_duration * coch_sr)



pad_dur = 0.5
# cue_crop_op = CenterCropCoch(crop_duration,
#                                 2.5,
#                                 coch_sr,
#                                 pad_dur=pad_dur).cuda()

center_crop = CenterCropCuePad(sig_duration, crop_duration, signal_sr, pad_dur=pad_dur).cuda() 


print(n_cue_frames)

config['model']['n_cue_frames'] = n_cue_frames
#
# module = binaural_lightning.BinauralAttentionModule.load_from_checkpoint(checkpoint_path=ckpt_path, config=config, strict=False)
coch_gram = cm.AttnAudioInputRepresentation(**config['audio']).cuda()

# int(torch.__version__.split('.')[0])
model = CueDurationCNNNew(**config['model'])

# reformat state dict to match model 
checkpoint = torch.load(ckpt_path)
checkpoint['state_dict'] = {k.replace('model._orig_mod.', ''):v for k,v in checkpoint['state_dict'].items()}
new_state_dict = reformat_state_dict(checkpoint, True)
norm_param_dict = {k:v for k,v in new_state_dict.items() if 'norm' in k}

# populate layer norm params 
for key, param in norm_param_dict.items():
    layer_name = key.split('.')[1]
    n_cue_frames_at_layer = model.layer_norm_params[f'{layer_name}']['ln_cue_frames']
    param_type = 'weight' if 'weight' in key else 'bias'
    model.layer_norm_params[f'{layer_name}'][f'{param_type}'] = param
    # take middle frames of cue for layer norm params
    start = int((param.shape[-1] - n_cue_frames_at_layer) / 2)
    end = start + n_cue_frames_at_layer
    model.layer_norm_params[f'{layer_name}'][f'cue_{param_type}'] = param[..., start : end]
    
model.load_state_dict(new_state_dict, strict=False)


model = model.eval().cuda()

with torch.no_grad(): 
    for ix, batch in enumerate(tqdm(dataloader)):
        if ix > 20:
            break
        cue, mixture, label, confusion = batch
        cue = center_crop(cue)
        cue, mixture = coch_gram(cue.cuda(), mixture.cuda())
        # cue = cue_crop_op(cue)
        # cue_mask_ixs = torch.arange(cue.shape[0])
        logits = model(cue, mixture, None)
        probs = logits.softmax(dim=-1).cpu().detach().numpy()

        # get top 2 probs for each example
        top_2_probs = torch.topk(logits.softmax(-1), 5, dim=-1).indices.cpu().detach().numpy()
        return_both = (np.isin(label.numpy(), top_2_probs) * np.isin(confusion.numpy(), top_2_probs)).astype('int')
        
        targ_probs = probs[torch.arange(probs.shape[0]), label]
        conf_probs = probs[torch.arange(probs.shape[0]), confusion]
        probs_of_interest = np.concatenate([targ_probs[:, None], conf_probs[:, None]], axis=1)

        preds = logits.softmax(dim=-1).argmax(dim=-1).cpu().detach().numpy().astype('int')
        true_word = label.numpy().astype('int')
        accuracy = (preds == true_word).astype('int')
        conf = (preds == confusion.numpy().astype('int')).astype('int')
        confusions_list.append(conf)
        accuracies.append(accuracy)
        pred_list.append(preds)
        true_word_int.append(true_word)
        all_probs_of_interest.append(probs_of_interest)
        guessed_both.append(return_both)
        
accuracies = np.concatenate(accuracies)
pred_list = np.concatenate(pred_list)
true_word_int = np.concatenate(true_word_int)
confusions_list = np.concatenate(confusions_list)
all_probs_of_interest = np.concatenate(all_probs_of_interest, axis=0)
guessed_both = np.concatenate(guessed_both)

output_dict['probs_of_interest'] = all_probs_of_interest
output_dict['results'] = accuracies
output_dict['preds'] = pred_list
output_dict['true_word_int'] = true_word_int

print(f"Accuracy using {crop_duration}s cue: {np.mean(accuracies):.2f}, ({stats.sem(accuracies):.2f})")
print(f"Confusions using {crop_duration}s cue: {np.mean(confusions_list):.2f}, ({stats.sem(confusions_list):.2f})")
print(f"Guessed both talker words: {np.mean(guessed_both):.2f}, ({stats.sem(guessed_both):.2f})")

Padding to duration
2500
center_crop=True
binaural=True
Binaural cochleagram
using FIR cochleagram
v08 True
num_classes={'num_words': 800}
Model performing word task
Using singe gain function per layer
Conv block order: LN -> Conv -> ReLU
fc_attn: True
coch_affine: True
Using cue duration of 2500 frames
Using cue duration of 617 frames
Using cue duration of 151 frames
Using cue duration of 30 frames
Using cue duration of 7 frames
Using cue duration of 2 frames
Using cue duration of 1 frames
Using cue duration of 1 frames
2500
Using cue duration of 2500 frames
Using cue duration of 617 frames
Using cue duration of 151 frames
Using cue duration of 30 frames
Using cue duration of 7 frames
Using cue duration of 2 frames
Using cue duration of 1 frames


 34%|███▍      | 21/61 [00:14<00:26,  1.50it/s]

Accuracy using 0.25s cue: 0.44, (0.03)
Confusions using 0.25s cue: 0.06, (0.01)
Guessed both talker words: 0.26, (0.02)


Using cue duration of 2 seconds    
Accuracy using fg as cue: 0.56, (0.01)    
Confusions using fg as cue: 0.09, (0.00)    
Guessed both talker words: 0.12, (0.01)    

Using cue duration of 0.1 seconds    
Accuracy using fg as cue: 0.50, (0.01)    
Confusions using fg as cue: 0.09, (0.01)    
Guessed both talker words: 0.11, (0.01)

In [ ]:
# print(f"Accuracy using {crop_duration}s cue: {np.mean(accuracies):.2f}, ({stats.sem(accuracies):.2f})")
# print(f"Confusions using {crop_duration}s cue: {np.mean(confusions_list):.2f}, ({stats.sem(confusions_list):.2f})")
# print(f"Guessed both talker words: {np.mean(guessed_both):.2f}, ({stats.sem(guessed_both):.2f})")

In [ ]:
# print(f"Using cue duration of {cue_duration} seconds")
# print(f"Accuracy using fg as cue: {np.mean(accuracies):.2f}, ({stats.sem(accuracies):.2f})")
# print(f"Confusions using fg as cue: {np.mean(confusions_list):.2f}, ({stats.sem(confusions_list):.2f})")
# print(f"Guessed both talker words: {np.mean(guessed_both):.2f}, ({stats.sem(guessed_both):.2f})")